# Week 03 Adapter Sprint: three-way held-out evaluation (BASE vs SFT vs DPO)

**Question:** did adaptation reduce unsupported address fabrication **while preserving correct use of supplied addresses**, rather than simply learning to output `null`?

**This is a controlled three-way model-state comparison, not a stock-inference benchmark.** All three states share one NF4 base, the same fp32 upcast, fp16 autocast, prompts, template, greedy decoding and order. Only the active adapter differs.

**Generations:** 180 primary generations (60 × 3) form all behavioral denominators. The 15 determinism-probe generations never enter them. The scoring script's SHA-256 is frozen in the config, and the run refuses to start if it has changed.

**Design:**
- The same 60 frozen held-out complaints (36 with an exact address, 24 without, 4 per no-address condition) are run in the same order.
- All three states use the same system prompt and chat template, greedy decoding with batch size 1, and the same frozen scoring rules (`scripts/eval_scoring.py`, whose SHA-256 is recorded).
- **BASE** is the NF4 base with adapters disabled. **SFT** and **DPO** are the verified adapters on that same base, switched in place. Nothing is trained, merged or overwritten.

All logic lives in `scripts/evaluate_models.py`. This notebook only calls repository scripts.

**Rules:**
- If any cell fails, **stop**.
- No numbers are pre-filled.
- Raw generations are never edited.
- DPO's training-set reward accuracy (0.9933) is **not** held-out evidence. This evaluation is the test.

## 1. Runtime / GPU check (before installing anything)

In [ ]:
!nvidia-smi
import sys, torch
print("python", sys.version.split()[0])
print("preinstalled torch", torch.__version__, "| built for CUDA", torch.version.cuda, "| CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("STOP: no CUDA GPU. Runtime > Change runtime type > T4 GPU, then restart from Section 1.")
print("GPU:", torch.cuda.get_device_name(0), "| capability:", torch.cuda.get_device_capability(0))
with open("/content/torch_constraint.txt", "w") as f:
    f.write(f"torch=={torch.__version__}\n")
print(open("/content/torch_constraint.txt").read())

## 2. Clone the repository (exact reviewed commit required)

In [ ]:
REPO_URL = "https://github.com/brentoatis-collab/week03-adapter-sprint.git"
REPO_DIR = "/content/week03-adapter-sprint"
EXPECTED_COMMIT = ""   # REQUIRED: the reviewed Step 6 commit

import os
if not EXPECTED_COMMIT:
    raise RuntimeError("STOP: set EXPECTED_COMMIT to the reviewed commit.")
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -5

## 3. Install pinned dependencies
A `pip check` line saying *ipython requires jedi* comes from Colab's own IPython and needs no action (README §6).

In [ ]:
%cd /content/week03-adapter-sprint
!pip install -r requirements.txt -c /content/torch_constraint.txt
!pip check || echo "pip check reported issues (see above) - review before continuing"

## 4. ⚠️ RESTART REQUIRED
**Runtime → Restart session**, then continue from **Section 5**.

## 5. Preflight on the exact commit

In [ ]:
import subprocess

def run(cmd):
    """Run a repository script, stream its output, and STOP the notebook on a non-zero exit."""
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"STOP: command failed with exit code {rc}: {' '.join(cmd)}")

import os, sys
os.chdir("/content/week03-adapter-sprint")
EXPECTED_COMMIT = ""   # REQUIRED again after the restart (same value as Section 2)
if not EXPECTED_COMMIT:
    raise RuntimeError("STOP: set EXPECTED_COMMIT.")
run([sys.executable, "training/preflight.py", "--require-cuda", "--expected-commit", EXPECTED_COMMIT])

## 6. Restore both adapters from Google Drive and verify them independently
For each adapter, the restored `adapter_model.safetensors` must hash to the value in `configs/adapter_config.json` (`evaluation.expected_*_adapter_sha256`) **and** match the logged training row. Any mismatch is a hard stop. The script checks these again, and also that the in-memory tensors equal the files bit-for-bit.

In [ ]:
from google.colab import drive
import json, shutil, hashlib, csv
drive.mount("/content/drive")
cfg = json.load(open("configs/adapter_config.json"))["evaluation"]
log = {r["run_id"]: r for r in csv.DictReader(open("logs/experiment_log.csv"))}
ADAPTERS = {}
for which, sub in (("sft", "qlora_sft"), ("dpo", "dpo")):
    run_id = cfg[f"{which}_run_id"]
    src = f"/content/drive/MyDrive/week03_adapter_sprint/{sub}/{run_id}"
    dst = f"outputs/{sub}/{run_id}"
    if not os.path.isdir(src):
        raise RuntimeError(f"STOP: {which} run directory not on Drive: {src}")
    if os.path.exists(dst):
        raise RuntimeError(f"STOP: {dst} already exists in this runtime; not overwriting.")
    shutil.copytree(src, dst)
    actual = hashlib.sha256(open(os.path.join(dst, "final_adapter", "adapter_model.safetensors"), "rb").read()).hexdigest()
    expected, logged = cfg[f"expected_{which}_adapter_sha256"], log[run_id]["adapter_sha256"]
    print(f"{which.upper()}: restored={actual}\n     expected={expected}\n     logged  ={logged}")
    if not (actual == expected == logged):
        raise RuntimeError(f"STOP: {which} adapter SHA-256 mismatch.")
    ADAPTERS[which] = os.path.join(dst, "final_adapter")
print("both adapters verified:", ADAPTERS)

## 7. Frozen eval data integrity (no regeneration)

In [ ]:
run([sys.executable, "scripts/audit_instruction_data.py"])
run(["git", "diff", "--exit-code", "--stat", "--", "data/"])
print("data/ matches the committed frozen datasets")

## 8. Dry run (strict)
Checks:
- the population and denominators (60 / 36 / 24; 4 per condition; street-only 2 actionable + 2 clarify)
- the scoring self-test: gold must score perfectly, and synthetic always-null / always-fabricate outputs must be caught
- both adapter SHA-256 gates
- the prompts and the greedy generation config

In [ ]:
run([sys.executable, "scripts/evaluate_models.py", "--sft-adapter-dir", ADAPTERS["sft"],
     "--dpo-adapter-dir", ADAPTERS["dpo"], "--dry-run"])

## 9. Formal evaluation: 3 states × 60 complaints
The run order is BASE → SFT → DPO, each over the frozen file order.

Look in the output for:
- the quantization evidence
- `adapters in memory == files`
- the switching probe: states differ, repeats are identical
- the effective generation config
- the per-state timing
- the summary table
- the determinism probe: the first 5 examples are regenerated per state and must be identical

Results are written to `results/three_way_eval/<eval_id>/`. **If this cell fails: STOP.**

In [ ]:
run([sys.executable, "scripts/evaluate_models.py", "--sft-adapter-dir", ADAPTERS["sft"],
     "--dpo-adapter-dir", ADAPTERS["dpo"]])

## 10. Results: summary, per-condition, per-example evidence

In [ ]:
import glob, json
import pandas as pd
from IPython.display import Markdown, display
RES = sorted(glob.glob("results/three_way_eval/three_way-*"))[-1]
print("results:", RES)
display(Markdown(open(f"{RES}/summary.md").read()))
agg = json.load(open(f"{RES}/summary.json"))
man = json.load(open(f"{RES}/manifest.json"))
print("determinism probe:", man["determinism_probe"], "| per-state:", man["per_state"])
print("scoring rules sha256:", man["scoring_rules_sha256"], "| eval data sha256:", man["eval_data_sha256"])
for st in ("base", "sft", "dpo"):
    print(f"\n{st.upper()} exact-address outcomes:", agg[st]["exact_address_outcomes"])
    print(f"{st.upper()} no-address outcomes   :", agg[st]["no_address_outcomes"])
    display(pd.DataFrame(agg[st]["no_address_by_condition"]).T.fillna(0).astype(int))
    print(f"{st.upper()} urgency by level:", agg[st]["urgency_by_level"], "| flags:", agg[st]["flag_counts"])

In [ ]:
# Side-by-side per-example evidence (for choosing recording examples; does not change any metric)
sc = pd.DataFrame([json.loads(l) for l in open(f"{RES}/scored.jsonl")])
gen = pd.DataFrame([json.loads(l) for l in open(f"{RES}/generations.jsonl")])
ev = pd.DataFrame([json.loads(l) for l in open("data/instruction_eval.jsonl")])
ev["gold_address"] = ev["response"].map(lambda r: json.loads(r)["address_or_null"])
wide = ev[["id", "location_type", "needs_clarification", "complaint", "gold_address"]].copy()
for st in ("base", "sft", "dpo"):
    s = sc[sc.state == st].set_index("id")
    g = gen[gen.state == st].set_index("id")
    wide[f"{st}_addr"] = wide.id.map(s.pred_address)
    wide[f"{st}_outcome"] = wide.id.map(s.address_outcome)
    wide[f"{st}_core_ok"] = wide.id.map(s.core_correct)
    wide[f"{st}_raw"] = wide.id.map(g.raw_text)
pd.set_option("display.max_colwidth", 80)
display(wide[["id", "location_type", "gold_address", "base_addr", "base_outcome", "sft_addr", "sft_outcome", "dpo_addr", "dpo_outcome"]])
changed = wide[(wide.sft_outcome != wide.dpo_outcome) | (wide.base_outcome != wide.sft_outcome)]
print(f"{len(changed)} examples whose address outcome changes across states")
display(changed[["id", "location_type", "complaint", "gold_address", "base_outcome", "sft_outcome", "dpo_outcome"]])

## 11. Persist results
`results/three_way_eval/<eval_id>/` is **tracked by git**. Commit that directory (and nothing else) and push it. Also copy it to Drive, with a SHA-256 check.

In [ ]:
from google.colab import drive
import shutil, hashlib, os
drive.mount("/content/drive")
dest = os.path.join("/content/drive/MyDrive/week03_adapter_sprint/three_way_eval", os.path.basename(RES))
os.makedirs(os.path.dirname(dest), exist_ok=True)
if os.path.exists(dest):
    raise RuntimeError(f"STOP: {dest} exists; not overwriting.")
shutil.copytree(RES, dest)
for f in sorted(os.listdir(RES)):
    a = hashlib.sha256(open(os.path.join(RES, f), "rb").read()).hexdigest()
    b = hashlib.sha256(open(os.path.join(dest, f), "rb").read()).hexdigest()
    print(f"{f:20s} {'MATCH' if a == b else 'MISMATCH'}")
    assert a == b
!git status --short

### Interpretation discipline (README §9)
- **Keep the claims separate:** (A) unsupported exact-address fabrication; the invalid non-null address field (contract); strict-JSON contract compliance; (B) retention; (C) over-null. A fabrication drop together with a large over-null rise is **not** an unqualified improvement.
- **Copied non-exact values** (block ranges, landmarks, intersections) count as invalid non-null, **not** as fabrication.
- **Exact retention is strict:** suffix variants are reported separately.
- **DPO's training-set reward accuracy is not held-out evidence.** The 0.663 digit-count shortcut is a known way DPO could "win" by avoiding numbers. The B and C rows are where that would show up.
- **Report urgency accuracy next to the majority-per-category baseline**, and every per-level figure with its `n`. `emergency` has only n = 3.
- **Summary exact match is diagnostic only**, not a primary metric.
- **Review flagged examples by hand before drawing conclusions from the affected metrics:** `address_suffix_variant`, `address_non_address_value`, `location_anchor_only`, `json_extracted_not_strict`, `category_is_other_mentioned_issue` and `no_eos_max_new_tokens_reached`.